# Module 3.5 — Is Bigger Actually Better?
**DeskMate SLM Curriculum · Phase 3 · Notebook 19**

**Recommended hardware:** Rented A100 40 GB (~$3–6 total for all three runs)

Three models, same SFT data, same gold eval set:
- `M_A` — Qwen2.5-1.5B QLoRA (from Module 3.4)
- `M_B` — Qwen2.5-1.5B full fine-tune (bf16, all params)
- `M_C` — Qwen2.5-7B QLoRA

Read `3.5_bigger_vs_smaller.md` for theory and the decision framework.

---

## Step 0 — Install

In [ ]:
%%capture
!pip install -q bitsandbytes==0.43.3 peft==0.12.0 trl==0.10.1 \
               transformers==4.44.0 datasets==2.21.0 torch==2.3.1 \
               accelerate==0.34.0 rouge-score==0.1.2

In [ ]:
import random, json, pathlib, time, os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_from_disk, Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, TaskType, PeftModel, get_peft_model
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from rouge_score import rouge_scorer as rs

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
HAS_GPU = device == 'cuda'
print(f'Device : {device}')
if HAS_GPU:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU    : {p.name}  ({p.total_memory/1e9:.1f} GB VRAM)')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS    = PROJECT_ROOT / 'reports'
ADAPTER_14 = MODELS_DIR / 'deskmate-qlora-adapter'   # from Module 3.4
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
print(f'Runtime : {RUNTIME}')

## Step 2 — Load SFT Dataset & Gold Eval Set

In [ ]:
sft_path = DATA_PROC / 'sft_dataset'
if sft_path.exists():
    sft_ds = load_from_disk(str(sft_path))
    print(f'SFT: train={len(sft_ds["train"])}  val={len(sft_ds["val"])}')
else:
    print('SFT dataset not found — building placeholder')
    INTENTS = ['account_access','billing_dispute','technical_bug','usage_question','outage_report']
    rows = []
    for i in range(300):
        intent = INTENTS[i % len(INTENTS)]
        rows.append({'ticket': f'I have a {intent.replace("_"," ")} issue #{i}.',
                     'reply':  f'Thank you for contacting DeskMate. We will resolve '
                               f'your {intent.replace("_"," ")} issue.',
                     'intent': intent, 'context': None})
    df = pd.DataFrame(rows)
    sft_ds = DatasetDict({'train': Dataset.from_pandas(df.iloc[:240]),
                          'val':   Dataset.from_pandas(df.iloc[240:])})

# Gold eval set: 50 held-out examples with reference replies
gold_path = DATA_PROC / 'gold.jsonl'
if gold_path.exists():
    gold_df = pd.read_json(gold_path, lines=True)
    if 'reference_reply' in gold_df.columns:
        gold_df = gold_df[gold_df['reference_reply'].notna()].head(50)
    else:
        gold_df = gold_df.head(50)
        gold_df['reference_reply'] = gold_df.get('reply', gold_df.get('text', ''))
else:
    print('No gold.jsonl — using val split as proxy eval set')
    val_rows = sft_ds['val'].to_pandas()
    val_rows['reference_reply'] = val_rows.get('reply', '')
    gold_df = val_rows.head(50)

print(f'Gold eval set: {len(gold_df)} examples')
print(f'Columns: {list(gold_df.columns)}')

## Step 3 — Shared Tokenizer & Inference Helper

In [ ]:
BASE_SMALL = 'Qwen/Qwen2.5-1.5B'
BASE_LARGE = 'Qwen/Qwen2.5-7B'
SYSTEM_MSG = 'You are DeskMate, a concise support assistant. Respond in 2-4 sentences.'
RESPONSE_TEMPLATE = '<|im_start|>assistant\n'

tokenizer = AutoTokenizer.from_pretrained(BASE_SMALL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def make_prompt(ticket, context=None):
    user = (('Context: ' + context + '\n\n') if context else '') + 'Ticket: ' + ticket
    msgs = [{'role':'system','content':SYSTEM_MSG},
            {'role':'user',  'content':user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def generate_reply(model, tok, ticket, context=None, max_new_tokens=150):
    prompt = make_prompt(ticket, context)
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    model.eval()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tok.eos_token_id)
    new_toks = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(new_toks, skip_special_tokens=True).strip()

print('Tokenizer loaded. Inference helper ready.')

## Step 4 — Train Model B: Qwen2.5-1.5B Full Fine-Tune

All parameters trained in bf16. No LoRA, no quantisation.
Requires ~12 GB VRAM — comfortable on A100 40 GB.

Set `TRAIN_FULL_FT = True` when on A100.

In [ ]:
TRAIN_FULL_FT = False   # Set True on A100

full_ft_path = MODELS_DIR / 'deskmate-1.5b-full-ft'

if TRAIN_FULL_FT and not full_ft_path.exists():
    print('Loading 1.5B in bf16 for full fine-tune ...')
    model_b = AutoModelForCausalLM.from_pretrained(
        BASE_SMALL, torch_dtype=torch.bfloat16,
        device_map='auto', trust_remote_code=True)
    model_b.config.use_cache = False

    collator = DataCollatorForCompletionOnlyLM(RESPONSE_TEMPLATE, tokenizer=tokenizer)
    args_b = TrainingArguments(
        output_dir=str(full_ft_path),
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        learning_rate=1e-5,
        warmup_ratio=0.05,
        lr_scheduler_type='cosine',
        bf16=True, logging_steps=25,
        eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True,
        report_to='none', seed=SEED,
    )
    trainer_b = SFTTrainer(
        model=model_b, args=args_b,
        train_dataset=sft_ds['train'], eval_dataset=sft_ds['val'],
        data_collator=collator, tokenizer=tokenizer, max_seq_length=512,
    )
    trainer_b.train()
    trainer_b.model.save_pretrained(str(full_ft_path))
    tokenizer.save_pretrained(str(full_ft_path))
    print(f'Full FT saved: {full_ft_path}')
    del model_b, trainer_b; torch.cuda.empty_cache()
elif full_ft_path.exists():
    print(f'Full FT checkpoint found: {full_ft_path}')
else:
    print('Skipped (TRAIN_FULL_FT=False). Set True on A100 and re-run.')

## Step 5 — Train Model C: Qwen2.5-7B QLoRA

NF4 + LoRA r=16. Requires ~8–10 GB VRAM. Comfortable on A100 40 GB.

Set `TRAIN_7B = True` when on A100.

In [ ]:
TRAIN_7B = False   # Set True on A100

qlora_7b_path = MODELS_DIR / 'deskmate-7b-qlora-adapter'

if TRAIN_7B and not qlora_7b_path.exists():
    print('Loading 7B in 4-bit ...')
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model_c = AutoModelForCausalLM.from_pretrained(
        BASE_LARGE, quantization_config=bnb,
        device_map='auto', trust_remote_code=True)
    model_c.config.use_cache = False
    model_c.enable_input_require_grads()
    lora_7b = LoraConfig(
        r=16, lora_alpha=32, target_modules='all-linear',
        lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM)
    model_c = get_peft_model(model_c, lora_7b)
    model_c.print_trainable_parameters()
    collator = DataCollatorForCompletionOnlyLM(RESPONSE_TEMPLATE, tokenizer=tokenizer)
    args_c = TrainingArguments(
        output_dir=str(qlora_7b_path),
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        learning_rate=2e-4, warmup_ratio=0.05,
        lr_scheduler_type='cosine',
        bf16=True, logging_steps=25,
        eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True,
        report_to='none', seed=SEED,
    )
    trainer_c = SFTTrainer(
        model=model_c, args=args_c,
        train_dataset=sft_ds['train'], eval_dataset=sft_ds['val'],
        data_collator=collator, tokenizer=tokenizer, max_seq_length=512,
    )
    trainer_c.train()
    trainer_c.model.save_pretrained(str(qlora_7b_path))
    tokenizer.save_pretrained(str(qlora_7b_path))
    print(f'7B adapter saved: {qlora_7b_path}')
    del model_c, trainer_c; torch.cuda.empty_cache()
elif qlora_7b_path.exists():
    print(f'7B adapter found: {qlora_7b_path}')
else:
    print('Skipped (TRAIN_7B=False). Set True on A100 and re-run.')

## Step 6 — Load All Three Models for Evaluation

In [ ]:
loaded_models = {}

# Model A: 1.5B QLoRA from Module 3.4
if ADAPTER_14.exists():
    print('Loading Model A: 1.5B QLoRA (Module 3.4) ...')
    if HAS_GPU:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                 bnb_4bit_compute_dtype=torch.bfloat16)
        base_a = AutoModelForCausalLM.from_pretrained(
            BASE_SMALL, quantization_config=bnb, device_map='auto', trust_remote_code=True)
    else:
        base_a = AutoModelForCausalLM.from_pretrained(
            BASE_SMALL, torch_dtype=torch.float32, trust_remote_code=True)
    loaded_models['1.5B QLoRA'] = PeftModel.from_pretrained(base_a, str(ADAPTER_14))
    print('  Model A loaded.')
else:
    print('Model A adapter not found — run Module 3.4 first.')

# Model B: 1.5B full fine-tune
if full_ft_path.exists():
    print('Loading Model B: 1.5B full FT ...')
    dtype = torch.bfloat16 if HAS_GPU else torch.float32
    loaded_models['1.5B Full FT'] = AutoModelForCausalLM.from_pretrained(
        str(full_ft_path), torch_dtype=dtype, device_map='auto', trust_remote_code=True)
    print('  Model B loaded.')
else:
    print('Model B not found — set TRAIN_FULL_FT=True and train first.')

# Model C: 7B QLoRA
if qlora_7b_path.exists():
    print('Loading Model C: 7B QLoRA ...')
    tok_7b = AutoTokenizer.from_pretrained(BASE_LARGE, trust_remote_code=True)
    if tok_7b.pad_token is None: tok_7b.pad_token = tok_7b.eos_token
    if HAS_GPU:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                 bnb_4bit_compute_dtype=torch.bfloat16)
        base_c = AutoModelForCausalLM.from_pretrained(
            BASE_LARGE, quantization_config=bnb, device_map='auto', trust_remote_code=True)
    else:
        base_c = AutoModelForCausalLM.from_pretrained(
            BASE_LARGE, torch_dtype=torch.float32, trust_remote_code=True)
    loaded_models['7B QLoRA'] = (PeftModel.from_pretrained(base_c, str(qlora_7b_path)), tok_7b)
    print('  Model C loaded.')
else:
    print('Model C not found — set TRAIN_7B=True and train first.')

print(f'\nLoaded: {list(loaded_models.keys())}')

## Step 7 — Generate Replies on Gold Eval Set

In [ ]:
# Use at most 50 gold examples
eval_rows = gold_df.head(50).to_dict('records')
predictions = {name: [] for name in loaded_models}

for name, m in loaded_models.items():
    tok = tokenizer
    model = m
    # 7B uses its own tokenizer (stored as tuple)
    if isinstance(m, tuple):
        model, tok = m
    print(f'Generating for {name} ...')
    for row in eval_rows:
        ticket  = row.get('ticket', row.get('text', ''))
        context = row.get('context')
        # 7B uses its own prompt template
        if isinstance(m, tuple):
            user = (('Context: ' + context + '\n\n') if context else '') + 'Ticket: ' + ticket
            msgs = [{'role':'system','content':SYSTEM_MSG}, {'role':'user','content':user}]
            prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
            model.eval()
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=150,
                                    do_sample=False, pad_token_id=tok.eos_token_id)
            new_toks = out[0][inputs['input_ids'].shape[1]:]
            reply = tok.decode(new_toks, skip_special_tokens=True).strip()
        else:
            reply = generate_reply(model, tok, ticket, context)
        predictions[name].append(reply)
    print(f'  Done: {len(predictions[name])} replies')

print('\nSample outputs (first example):')
if eval_rows:
    print(f'Ticket: {eval_rows[0].get("ticket", eval_rows[0].get("text",""))}')
    for name, preds in predictions.items():
        print(f'{name}: {preds[0][:120]}...' if preds and len(preds[0]) > 120 else f'{name}: {preds[0] if preds else "N/A"}')

## Step 8 — ROUGE-L Scores

In [ ]:
scorer = rs.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = {}

references = [r.get('reference_reply', r.get('reply', '')) for r in eval_rows]

for name, preds in predictions.items():
    if not preds: continue
    scores = [scorer.score(ref, pred)['rougeL'].fmeasure
              for ref, pred in zip(references, preds)]
    rouge_scores[name] = scores
    print(f'{name:<20}  ROUGE-L mean={np.mean(scores):.3f}  '
           f'median={np.median(scores):.3f}  std={np.std(scores):.3f}')

# Plot distributions
if rouge_scores:
    fig, ax = plt.subplots(figsize=(9, 4))
    for i, (name, scores) in enumerate(rouge_scores.items()):
        ax.hist(scores, bins=20, alpha=0.6, label=name)
    ax.set_xlabel('ROUGE-L F1'); ax.set_ylabel('Count')
    ax.set_title('ROUGE-L Distribution by Model')
    ax.legend(); plt.tight_layout(); plt.show()

## Step 9 — Pairwise LLM-as-Judge

Compare Model A (1.5B QLoRA) vs Model C (7B QLoRA) pairwise.
Randomise which is 'A' vs 'B' in each prompt to mitigate position bias.

In [ ]:
import anthropic

def judge_pair(ticket, reply_x, reply_y, client, model='claude-haiku-4-5-20251001'):
    # Randomly assign which reply is presented as A vs B
    flip = random.random() < 0.5
    a, b = (reply_y, reply_x) if flip else (reply_x, reply_y)
    prompt = (
        'You are evaluating customer support ticket replies. '
        'Given the ticket below, which reply is better? '
        'Reply with exactly one word: A, B, or TIE.\n\n'
        'Ticket: ' + ticket + '\n\nReply A: ' + a + '\n\nReply B: ' + b + '\n\nVerdict:'
    )
    resp = client.messages.create(
        model=model, max_tokens=5,
        messages=[{'role':'user','content':prompt}]
    )
    verdict = resp.content[0].text.strip().upper()
    if verdict not in ('A','B','TIE'): verdict = 'TIE'
    # Unflip: if we swapped, A-win means y-win
    if flip and verdict == 'A': return 'B'
    if flip and verdict == 'B': return 'A'
    return verdict

RUN_JUDGE = False   # Set True to run pairwise evaluation (uses Claude API)
judge_results = []

if RUN_JUDGE and '1.5B QLoRA' in predictions and '7B QLoRA' in predictions:
    import os
    hf_token = os.environ.get('ANTHROPIC_API_KEY') or ''
    if not hf_token:
        try:
            from google.colab import userdata
            hf_token = userdata.get('ANTHROPIC_API_KEY')
        except Exception:
            pass
    client = anthropic.Anthropic(api_key=hf_token)
    N_JUDGE = min(20, len(eval_rows))
    wins_a = wins_c = ties = 0
    for i in range(N_JUDGE):
        ticket  = eval_rows[i].get('ticket', eval_rows[i].get('text', ''))
        pred_a  = predictions['1.5B QLoRA'][i]
        pred_c  = predictions['7B QLoRA'][i]
        verdict = judge_pair(ticket, pred_a, pred_c, client)
        judge_results.append({'ticket': ticket, 'verdict': verdict})
        if verdict == 'A': wins_a += 1
        elif verdict == 'B': wins_c += 1
        else: ties += 1
    print(f'LLM-as-judge ({N_JUDGE} pairs):')
    print(f'  1.5B QLoRA wins : {wins_a} ({wins_a/N_JUDGE*100:.0f}%)')
    print(f'  7B QLoRA wins   : {wins_c} ({wins_c/N_JUDGE*100:.0f}%)')
    print(f'  Ties            : {ties} ({ties/N_JUDGE*100:.0f}%)')
else:
    print('Judge skipped (RUN_JUDGE=False or missing predictions).')
    print('Set RUN_JUDGE=True with ANTHROPIC_API_KEY in Colab secrets.')

## Step 10 — Latency Benchmark (p50 / p95)

In [ ]:
SAMPLE_TICKET = 'I cannot log in — my password reset email never arrived.'
N_WARMUP = 3
N_BENCH  = 15
latency_results = {}

for name, m in loaded_models.items():
    tok   = tokenizer
    model = m
    if isinstance(m, tuple):
        model, tok = m
    times = []
    # warmup
    for _ in range(N_WARMUP):
        _ = generate_reply(model, tok, SAMPLE_TICKET) if not isinstance(m, tuple) else None
    # benchmark
    for _ in range(N_BENCH):
        t0 = time.perf_counter()
        if isinstance(m, tuple):
            user  = 'Ticket: ' + SAMPLE_TICKET
            msgs  = [{'role':'system','content':SYSTEM_MSG},{'role':'user','content':user}]
            prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp = tok(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
            model.eval()
            with torch.no_grad():
                out = model.generate(**inp, max_new_tokens=150,
                                    do_sample=False, pad_token_id=tok.eos_token_id)
        else:
            generate_reply(model, tok, SAMPLE_TICKET)
        times.append((time.perf_counter() - t0) * 1000)
    p50 = np.percentile(times, 50)
    p95 = np.percentile(times, 95)
    latency_results[name] = {'p50_ms': p50, 'p95_ms': p95}
    print(f'{name:<20}  p50={p50:.0f}ms  p95={p95:.0f}ms')

## Step 11 — Cost Estimate ($/1k Requests)

In [ ]:
GPU_HOURLY = 0.75   # A10G estimate; adjust for your GPU

def cost_per_1k(p50_ms, gpu_hourly=GPU_HOURLY):
    requests_per_hour = 3_600_000 / p50_ms
    return gpu_hourly / requests_per_hour * 1000

print(f'Cost estimates (GPU rate: ${GPU_HOURLY}/hr):')
cost_results = {}
for name, lat in latency_results.items():
    c = cost_per_1k(lat['p50_ms'])
    cost_results[name] = c
    daily = c * 100   # 100k requests / day
    print(f'{name:<20}  ${c:.4f}/1k req  (${daily:.0f}/day at 100k req/day)')

## Step 12 — Three-Way Comparison Table

In [ ]:
print(f'{"Model":<20} {"ROUGE-L":>9} {"p50 ms":>8} {"p95 ms":>8} {"$/1k":>9}')
print('-' * 60)
all_names = list(set(list(rouge_scores.keys()) + list(latency_results.keys())))
table_rows = []
for name in ['1.5B QLoRA', '1.5B Full FT', '7B QLoRA']:
    rl  = f'{np.mean(rouge_scores[name]):.3f}' if name in rouge_scores else 'N/A'
    p50 = f'{latency_results[name]["p50_ms"]:.0f}' if name in latency_results else 'N/A'
    p95 = f'{latency_results[name]["p95_ms"]:.0f}' if name in latency_results else 'N/A'
    cost = f'{cost_results[name]:.4f}' if name in cost_results else 'N/A'
    print(f'{name:<20} {rl:>9} {p50:>8} {p95:>8} {cost:>9}')
    table_rows.append({'Model': name, 'ROUGE-L': rl, 'p50_ms': p50,
                       'p95_ms': p95, 'cost_per_1k': cost})

## Step 13 — Auto-Generate Decision Paragraph

In [ ]:
def auto_decision(rouge_scores, latency_results, cost_results):
    has_a = '1.5B QLoRA'  in rouge_scores and '1.5B QLoRA'  in latency_results
    has_c = '7B QLoRA'    in rouge_scores and '7B QLoRA'    in latency_results

    if not (has_a and has_c):
        return (
            'Incomplete data: not all three models were trained and evaluated. '
            'Re-run after setting TRAIN_FULL_FT=True and TRAIN_7B=True on A100.'
        )

    rl_a   = np.mean(rouge_scores['1.5B QLoRA'])
    rl_c   = np.mean(rouge_scores['7B QLoRA'])
    gap    = rl_c - rl_a
    cost_a = cost_results.get('1.5B QLoRA', 0)
    cost_c = cost_results.get('7B QLoRA', 0)
    lat_a  = latency_results['1.5B QLoRA']['p50_ms']
    lat_c  = latency_results['7B QLoRA']['p50_ms']
    cost_multiplier = cost_c / cost_a if cost_a > 0 else 0

    if gap < 0.03:
        verdict = (
            f'SHIP THE 1.5B MODEL. The ROUGE-L gap between 7B and 1.5B QLoRA is only '
            f'{gap:.3f} ({gap*100:.1f} points) — well within measurement noise for '
            f'generation tasks. The 7B model costs {cost_multiplier:.1f}x more per '
            f'request (${cost_c:.4f} vs ${cost_a:.4f}/1k) and runs {lat_c/lat_a:.1f}x '
            f'slower (p50: {lat_c:.0f}ms vs {lat_a:.0f}ms). This is not a trade-off '
            f'worth making for a narrow support-reply task. Invest in data quality '
            f'before model size.'
        )
    elif gap < 0.10:
        verdict = (
            f'INVESTIGATE BEFORE DECIDING. The ROUGE-L gap is {gap:.3f} ({gap*100:.1f} '
            f'points). This is non-trivial but not conclusive for generation tasks. '
            f'Run the pairwise judge (Step 9) to identify whether the gap represents '
            f'systematic failures (complex tickets, multi-step reasoning) or random '
            f'surface variation. If failures cluster on complex tickets, more SFT data '
            f'targeting those patterns will likely close the gap at zero inference cost premium.'
        )
    else:
        verdict = (
            f'CONSIDER THE 7B MODEL. The ROUGE-L gap is {gap:.3f} ({gap*100:.1f} '
            f'points) — large enough that the task complexity may exceed what a '
            f'fine-tuned 1.5B model can handle. Before paying {cost_multiplier:.1f}x '
            f'more per request, examine whether the task scope can be narrowed '
            f'(shorter replies, more structured output) to bring it within 1.5B range.'
        )
    return verdict

decision = auto_decision(rouge_scores, latency_results, cost_results)
print('DECISION:')
print(decision)

## Step 14 — Save Report

In [ ]:
lines = ['# Bigger vs Smaller: 3-Way Comparison Report\n']
lines.append('| Model | ROUGE-L | p50 (ms) | p95 (ms) | $/1k req |')
lines.append('|---|---|---|---|---|')
for row in table_rows:
    lines.append(
        f'| {row["Model"]} | {row["ROUGE-L"]} | {row["p50_ms"]} | '
        f'{row["p95_ms"]} | {row["cost_per_1k"]} |'
    )
lines.append('')
lines.append('## Decision\n')
lines.append(decision)
if judge_results:
    n = len(judge_results)
    wins_a = sum(1 for r in judge_results if r['verdict']=='A')
    wins_c = sum(1 for r in judge_results if r['verdict']=='B')
    ties   = n - wins_a - wins_c
    lines.append(f'\n## LLM Judge ({n} pairs)')
    lines.append(f'- 1.5B QLoRA wins: {wins_a} ({wins_a/n*100:.0f}%)')
    lines.append(f'- 7B QLoRA wins  : {wins_c} ({wins_c/n*100:.0f}%)')
    lines.append(f'- Ties           : {ties} ({ties/n*100:.0f}%)')

report_path = REPORTS / 'bigger_vs_smaller_report.md'
report_path.write_text('\n'.join(lines))
print(f'Report saved: {report_path}')

## Checkpoint

> **At what quality gap would the bigger model become worth its cost for DeskMate?**

Write your answer below.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| 3-way comparison report | `reports/bigger_vs_smaller_report.md` |
| 1.5B full FT checkpoint | `models/deskmate-1.5b-full-ft/` |
| 7B QLoRA adapter | `models/deskmate-7b-qlora-adapter/` |

**What you've built:** evidence-based justification for DeskMate's model choice.

**Next:** Module 3.6 — Evaluating generative output with a proper eval harness:
ROUGE-L, LLM-as-judge with bias mitigation, and a regression test set.